In [ ]:
import pandas as pd

In [ ]:
nvdimp_brbq = pd.read_csv('./raw/data/nvdimp_brbqplots_info.csv')
nvdimp_trad = pd.read_csv('./raw/data/nvdimp_tradplots_info.csv')
nvdimp_subplot = pd.read_csv('./raw/data/nvdimp_subplot.csv')

In [ ]:
# --- 1. 處理 brbq（有完整日期）---------------------------------
brbq_site = (
    nvdimp_brbq
    [['plotid', 'ddx', 'ddy', 'pdate']]        # ddx, ddy = 經緯度
    .copy()
)

brbq_site['date'] = pd.to_datetime(brbq_site['pdate'])
brbq_site['year'] = brbq_site['date'].dt.year
brbq_site['month'] = brbq_site['date'].dt.month
brbq_site['day'] = brbq_site['date'].dt.day

# --- 2. 處理 trad（只有 survey_year，當成該年 1 月 1 日）----------
trad_site = (
    nvdimp_trad
    [['plotid', 'ddx', 'ddy', 'survey_year']]
    .copy()
)

trad_site['year'] = trad_site['survey_year'].astype(int)
trad_site['date'] = pd.to_datetime(trad_site['year'].astype(str) + '-01-01')
trad_site['month'] = trad_site['date'].dt.month   # 會是 1
trad_site['day'] = trad_site['date'].dt.day     # 會是 1

# --- 3. 合併兩個 site 資料表 ------------------------------------
site_all = pd.concat(
    [
        brbq_site[['plotid', 'ddx', 'ddy', 'day', 'month', 'year']],
        trad_site[['plotid', 'ddx', 'ddy', 'day', 'month', 'year']]
    ],
    ignore_index=True
)

# --- 4. 把 site 資訊接到 subplot（物種名在 scientificName）--------
subplot_all = nvdimp_subplot.merge(site_all, on='plotid', how='left')

# 假設 ddx = longitude, ddy = latitude
result = subplot_all.rename(
    columns={
        'scientificName': 'species',
        'ddy': 'decimalLatitude',
        'ddx': 'decimalLongitude'
    }
)[['species', 'decimalLatitude', 'decimalLongitude', 'day', 'month', 'year']]

# result 就是你要的 dataframe

In [ ]:
result.to_csv('./raw/NVDIMP_raw.csv', index=None)